In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
#Funciones auxiliares sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold #Split y cross Validation
from sklearn.metrics import cohen_kappa_score, accuracy_score, balanced_accuracy_score #Metricas
from sklearn.utils import shuffle 


#Gradient Boosting
import lightgbm as lgb


# Configuramos el estilo de las visualizaciones
sns.set_style("whitegrid")
plt.style.use("fivethirtyeight")

breed_df = pd.read_csv(os.path.join(r"C:\Users\fchiavassa\Desktop\Nueva carpeta\Inteligencia Artificial\Codigos\Maestria\breed_labels.csv"))
color_df = pd.read_csv(os.path.join(r"C:\Users\fchiavassa\Desktop\Nueva carpeta\Inteligencia Artificial\Codigos\Maestria\color_labels.csv"))
state_df = pd.read_csv(os.path.join(r"C:\Users\fchiavassa\Desktop\Nueva carpeta\Inteligencia Artificial\Codigos\Maestria\state_labels.csv"))

# Definimos la ruta
ruta_archivo = r"C:\Users\fchiavassa\Desktop\Nueva carpeta\Inteligencia Artificial\Codigos\Maestria\train.csv"

# Intentamos leer con latin-1

# Agregamos el parámetro sep=';'
train_df = pd.read_csv(ruta_archivo, sep=',', encoding='latin-1')


In [18]:
SEED = 42 #Semilla de procesos aleatorios (para poder replicar exactamente al volver a correr un modelo)
TEST_SIZE = 0.2 #Facción para train/test= split

In [19]:
#Separo un 20% para test estratificado opr target
df_train, df_test = train_test_split(train_df,
                               test_size = TEST_SIZE,
                               random_state = SEED,
                               stratify = train_df.AdoptionSpeed)

In [16]:
import pandas as pd

# 1. Cargar el dataset original
# Asegúrate de que el archivo esté en la misma carpeta que tu notebook

def categorizar_tamano(row):
    breed = str(row['BreedName']).lower()
    pet_type = row['Type']
    
    # Gatos (Type = 2): Los clasificamos como 'Pequeño' por defecto en comparación a un perro
    if pet_type == 2:
        return 'Pequeño'
        
    # Perros (Type = 1)
    if 'mixed breed' in breed:
        return 'Mediano' # Promedio estadístico seguro para perros mestizos
        
    # A. Excepciones específicas (se evalúan antes que las reglas generales)
    excepciones_grandes = ['black russian terrier', 'airedale terrier', 'afghan hound', 'bloodhound', 'greyhound', 'irish wolfhound']
    excepciones_medianos = ['american staffordshire terrier', 'pit bull terrier', 'bull terrier', 'beagle', 'basset hound', 'whippet', 'standard schnauzer', 'cocker spaniel']
    excepciones_pequenos = ['dachshund', 'miniature schnauzer', 'toy', 'miniature']
    
    if any(exc in breed for exc in excepciones_grandes):
        return 'Grande'
    if any(exc in breed for exc in excepciones_medianos):
        return 'Mediano'
    if any(exc in breed for exc in excepciones_pequenos):
        return 'Pequeño'
        
    # B. Reglas generales por familias funcionales / palabras clave
    # La mayoría de los terriers y perros de compañía son pequeños
    keywords_pequenos = ['terrier', 'chihuahua', 'pomeranian', 'pug', 'shih tzu', 'maltese', 'pekingese', 'corgi', 'bichon', 'french bulldog', 'papillon', 'havanese', 'pinscher']
    
    # La mayoría de los pastores, cobradores, mastines y sabuesos son grandes
    keywords_grandes = ['retriever', 'shepherd', 'mastiff', 'great', 'husky', 'malamute', 'akita', 'hound', 'pointer', 'setter', 'collie', 'rottweiler', 'doberman', 'boxer', 'american bulldog', 'sheepdog', 'dane', 'bernard', 'pyrenees', 'newfoundland', 'weimaraner', 'ridgeback', 'corso', 'komondor', 'kuvasz', 'borzoi', 'dalmatian']
    
    if any(kw in breed for kw in keywords_pequenos):
        return 'Pequeño'
    if any(kw in breed for kw in keywords_grandes):
        return 'Grande'
        
    # C. Todo lo que no caiga en las reglas anteriores se asume Mediano (ej. Spaniels, Bulldogs ingleses, Spitz)
    return 'Mediano'

# 2. Aplicar la función para crear la nueva columna
breed_df['SizeCategory'] = breed_df.apply(categorizar_tamano, axis=1)

# Unimos indicando que Breed1 equivale a BreedID
df_train = df_train.merge(
    breed_df[['BreedID', 'SizeCategory']], 
    left_on='Breed1', 
    right_on='BreedID', 
    how='left'
)

df_test = df_test.merge(
    breed_df[['BreedID', 'SizeCategory']], 
    left_on='Breed1', 
    right_on='BreedID', 
    how='left'
)

# Opcional: eliminar la columna BreedID duplicada que queda tras el merge
df_train = df_train.drop(columns=['BreedID'])
df_test = df_test.drop(columns=['BreedID'])
# 3. Exportar el resultado a un nuevo archivo CSV
nuevo_archivo = 'PetFinder-BreedLabels-Sized.csv'
breed_df.to_csv(nuevo_archivo, index=False)

print(f"Archivo generado exitosamente: '{nuevo_archivo}'")
print("\nMuestra de cómo quedaron las categorías:")
display(breed_df.sample(10, random_state=42))

Archivo generado exitosamente: 'PetFinder-BreedLabels-Sized.csv'

Muestra de cómo quedaron las categorías:


,BreedID,Type,BreedName,SizeCategory
183,184,1,Pumi,Mediano
60,61,1,Chinese Crested Dog,Mediano
124,125,1,Irish Wolfhound,Grande
93,94,1,Fila Brasileiro,Mediano
63,64,1,Chocolate Labrador Retriever,Grande
9,10,1,American Staffordshire Terrier,Mediano
147,148,1,Manchester Terrier,Pequeño
158,159,1,Norfolk Terrier,Pequeño
168,169,1,Pekingese,Pequeño
33,34,1,Bloodhound,Grande


In [27]:
#Genero dataframes de train y test con sus respectivos targets
X_train = df_train.drop(columns=['AdoptionSpeed','RescuerID'])
y_train = df_train['AdoptionSpeed']

X_test = df_test.drop(columns=['AdoptionSpeed','RescuerID'])
y_test = df_test['AdoptionSpeed']

In [28]:
# 1. Definir explícitamente qué variables son categóricas
categorical_cols = ['Type', 'Breed1', 'Breed2', 'Gender', 'Color1', 'Color2', 'Color3', 
                    'Vaccinated', 'Dewormed', 'Sterilized', 'Health', 'State','Name','PetID','Description']


# Asegurarse de que el modelo entienda que son categorías (convertir a tipo 'category' en pandas)
for col in categorical_cols:
    X_train[col] = X_train[col].astype('category')
    X_test[col] = X_test[col].astype('category')

# 2. Configurar K-Fold Cross Validation
N_SPLITS = 5
folds = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

# Arreglos para guardar predicciones (Out-Of-Fold y Test)
oof_preds = np.zeros((len(X_train), 5)) # 5 es el número de clases
test_preds = np.zeros((len(X_test), 5))

# Nuevos parámetros base más controlados
lgb_params = {
    'objective': 'multiclass',
    'num_class': 5,
    'metric': 'multi_error',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': -1,
    'feature_fraction': 0.8,
    'seed': SEED,
    'verbose': -1
}

print("Iniciando entrenamiento con 5-Fold Cross Validation...")

for fold_, (trn_idx, val_idx) in enumerate(folds.split(X_train, y_train)):
    print(f"\\n--- Fold {fold_ + 1} ---")
    
    # Separar datos del fold
    X_tr, y_tr = X_train.iloc[trn_idx], y_train.iloc[trn_idx]
    X_val, y_val = X_train.iloc[val_idx], y_train.iloc[val_idx]
    
    # Crear Datasets de LightGBM
    trn_data = lgb.Dataset(X_tr, label=y_tr, categorical_feature=categorical_cols)
    val_data = lgb.Dataset(X_val, label=y_val, categorical_feature=categorical_cols, reference=trn_data)
    
    # Entrenar modelo usando callbacks modernos
    clf = lgb.train(
        lgb_params,
        trn_data,
        num_boost_round=1000,
        valid_sets=[trn_data, val_data],
        callbacks=[
            lgb.early_stopping(stopping_rounds=50, verbose=False),
            lgb.log_evaluation(period=100)
        ]
    )
    
    # Predecir sobre validación (Out-Of-Fold)
    oof_preds[val_idx] = clf.predict(X_val, num_iteration=clf.best_iteration)
    
    # Predecir sobre test (promediando los 5 folds)
    test_preds += clf.predict(X_test, num_iteration=clf.best_iteration) / N_SPLITS

# 3. Evaluar el modelo ensamblado
oof_predictions_classes = oof_preds.argmax(axis=1)
test_predictions_classes = test_preds.argmax(axis=1)

kappa_cv = cohen_kappa_score(y_train, oof_predictions_classes, weights='quadratic')
kappa_test = cohen_kappa_score(y_test, test_predictions_classes, weights='quadratic')

print(f"\\nScore Kappa de Validación Cruzada (OOF): {kappa_cv:.4f}")
print(f"Score Kappa final en Test Set: {kappa_test:.4f}")

Iniciando entrenamiento con 5-Fold Cross Validation...
\n--- Fold 1 ---
[100]	training's multi_error: 0.410839	valid_1's multi_error: 0.580242
\n--- Fold 2 ---
[100]	training's multi_error: 0.40938	valid_1's multi_error: 0.588579
[200]	training's multi_error: 0.326212	valid_1's multi_error: 0.588995
\n--- Fold 3 ---
\n--- Fold 4 ---
[100]	training's multi_error: 0.404794	valid_1's multi_error: 0.601917
[200]	training's multi_error: 0.321939	valid_1's multi_error: 0.596499
\n--- Fold 5 ---
[100]	training's multi_error: 0.409337	valid_1's multi_error: 0.577148
\nScore Kappa de Validación Cruzada (OOF): 0.3629
Score Kappa final en Test Set: 0.3082


In [29]:
import optuna
import lightgbm as lgb
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score

# Asegúrate de tener X_train, y_train y categorical_cols definidos previamente

def objective(trial):
    # 1. Definir el espacio de búsqueda de hiperparámetros
    param = {
    'objective': 'multiclass',
    'num_class': 5,
    'metric': 'multi_error',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': -1,
    'feature_fraction': 0.8,
    'seed': SEED,
    'verbose': -1,
    'learning_rate': trial.suggest_float('learning_rate', 0.05, 0.5, log=True),
    'num_leaves': trial.suggest_int('num_leaves', 10, 200),
    'max_depth': trial.suggest_int('max_depth', 7, 100),
    'feature_fraction': trial.suggest_float('feature_fraction', 0.4, 1.0),   
    }

    # 2. Configurar la Validación Cruzada
    N_SPLITS = 5
    folds = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
    oof_preds = np.zeros((len(X_train), 5))

    # 3. Bucle de entrenamiento por Fold
    for fold_, (trn_idx, val_idx) in enumerate(folds.split(X_train, y_train)):
        X_tr, y_tr = X_train.iloc[trn_idx], y_train.iloc[trn_idx]
        X_val, y_val = X_train.iloc[val_idx], y_train.iloc[val_idx]
        
        trn_data = lgb.Dataset(X_tr, label=y_tr, categorical_feature=categorical_cols)
        val_data = lgb.Dataset(X_val, label=y_val, categorical_feature=categorical_cols, reference=trn_data)
        
        # Entrenar el modelo
        clf = lgb.train(
            param,
            trn_data,
            num_boost_round=500, # Un número alto, early_stopping lo detendrá antes
            valid_sets=[trn_data, val_data],
            callbacks=[
                lgb.early_stopping(stopping_rounds=30, verbose=False)
            ]
        )
        
        # Guardar predicciones de validación
        oof_preds[val_idx] = clf.predict(X_val, num_iteration=clf.best_iteration)

    # 4. Calcular la métrica final (QWK) para este Trial
    oof_predictions_classes = oof_preds.argmax(axis=1)
    kappa_score = cohen_kappa_score(y_train, oof_predictions_classes, weights='quadratic')
    
    return kappa_score

# --- EJECUCIÓN DE LA OPTIMIZACIÓN ---

print("Iniciando búsqueda de hiperparámetros con Optuna...")

# Queremos MAXIMIZAR el score Kappa
study = optuna.create_study(direction='maximize', study_name="Petfinder_LGBM_Optuna")

# Ejecutar el estudio (puedes cambiar n_trials dependiendo del tiempo que tengas)
study.optimize(objective, n_trials=20)

# Mostrar resultados
print("\n--- ¡Optimización Terminada! ---")
print(f"Mejor Score Kappa (OOF): {study.best_value:.4f}")
print("Mejores Hiperparámetros encontrados:")
for key, value in study.best_params.items():
    print(f"    '{key}': {value}")
    

[I 2026-04-09 20:02:53,358] A new study created in memory with name: Petfinder_LGBM_Optuna


Iniciando búsqueda de hiperparámetros con Optuna...


[I 2026-04-09 20:03:10,338] Trial 0 finished with value: 0.35737471250497965 and parameters: {'learning_rate': 0.05044902559051097, 'num_leaves': 97, 'max_depth': 77, 'feature_fraction': 0.6393665305211824}. Best is trial 0 with value: 0.35737471250497965.
[I 2026-04-09 20:03:18,993] Trial 1 finished with value: 0.35669547707180205 and parameters: {'learning_rate': 0.2728503638624756, 'num_leaves': 32, 'max_depth': 48, 'feature_fraction': 0.5650039450823354}. Best is trial 0 with value: 0.35737471250497965.
[I 2026-04-09 20:03:34,400] Trial 2 finished with value: 0.35006065333755765 and parameters: {'learning_rate': 0.23438192157488494, 'num_leaves': 113, 'max_depth': 66, 'feature_fraction': 0.9382943564458955}. Best is trial 0 with value: 0.35737471250497965.
[I 2026-04-09 20:03:46,380] Trial 3 finished with value: 0.3480563830567952 and parameters: {'learning_rate': 0.1932535321420127, 'num_leaves': 72, 'max_depth': 80, 'feature_fraction': 0.7356734961864275}. Best is trial 0 with va

KeyboardInterrupt: 

In [ ]:
import pandas as pd

# 1. Cargar el dataset original
# Asegúrate de que el archivo esté en la misma carpeta que tu notebook

def categorizar_tamano(row):
    breed = str(row['BreedName']).lower()
    pet_type = row['Type']
    
    # Gatos (Type = 2): Los clasificamos como 'Pequeño' por defecto en comparación a un perro
    if pet_type == 2:
        return 'Pequeño'
        
    # Perros (Type = 1)
    if 'mixed breed' in breed:
        return 'Mediano' # Promedio estadístico seguro para perros mestizos
        
    # A. Excepciones específicas (se evalúan antes que las reglas generales)
    excepciones_grandes = ['black russian terrier', 'airedale terrier', 'afghan hound', 'bloodhound', 'greyhound', 'irish wolfhound']
    excepciones_medianos = ['american staffordshire terrier', 'pit bull terrier', 'bull terrier', 'beagle', 'basset hound', 'whippet', 'standard schnauzer', 'cocker spaniel']
    excepciones_pequenos = ['dachshund', 'miniature schnauzer', 'toy', 'miniature']
    
    if any(exc in breed for exc in excepciones_grandes):
        return 'Grande'
    if any(exc in breed for exc in excepciones_medianos):
        return 'Mediano'
    if any(exc in breed for exc in excepciones_pequenos):
        return 'Pequeño'
        
    # B. Reglas generales por familias funcionales / palabras clave
    # La mayoría de los terriers y perros de compañía son pequeños
    keywords_pequenos = ['terrier', 'chihuahua', 'pomeranian', 'pug', 'shih tzu', 'maltese', 'pekingese', 'corgi', 'bichon', 'french bulldog', 'papillon', 'havanese', 'pinscher']
    
    # La mayoría de los pastores, cobradores, mastines y sabuesos son grandes
    keywords_grandes = ['retriever', 'shepherd', 'mastiff', 'great', 'husky', 'malamute', 'akita', 'hound', 'pointer', 'setter', 'collie', 'rottweiler', 'doberman', 'boxer', 'american bulldog', 'sheepdog', 'dane', 'bernard', 'pyrenees', 'newfoundland', 'weimaraner', 'ridgeback', 'corso', 'komondor', 'kuvasz', 'borzoi', 'dalmatian']
    
    if any(kw in breed for kw in keywords_pequenos):
        return 'Pequeño'
    if any(kw in breed for kw in keywords_grandes):
        return 'Grande'
        
    # C. Todo lo que no caiga en las reglas anteriores se asume Mediano (ej. Spaniels, Bulldogs ingleses, Spitz)
    return 'Mediano'

# 2. Aplicar la función para crear la nueva columna
breed_df['SizeCategory'] = breed_df.apply(categorizar_tamano, axis=1)

# Unimos indicando que Breed1 equivale a BreedID
df_train = df_train.merge(
    breed_df[['BreedID', 'SizeCategory']], 
    left_on='Breed1', 
    right_on='BreedID', 
    how='left'
)

df_test = df_test.merge(
    breed_df[['BreedID', 'SizeCategory']], 
    left_on='Breed1', 
    right_on='BreedID', 
    how='left'
)

# Opcional: eliminar la columna BreedID duplicada que queda tras el merge
df_train = df_train.drop(columns=['BreedID'])
df_test = df_test.drop(columns=['BreedID'])
# 3. Exportar el resultado a un nuevo archivo CSV
nuevo_archivo = 'PetFinder-BreedLabels-Sized.csv'
breed_df.to_csv(nuevo_archivo, index=False)

print(f"Archivo generado exitosamente: '{nuevo_archivo}'")
print("\nMuestra de cómo quedaron las categorías:")
display(breed_df.sample(10, random_state=42))

In [91]:
# Creamos un diccionario para acumular las nuevas columnas rápidamente
nuevas_features = {}

# 1. Interacciones de Edad y Atributos Físicos
# Una mascota vieja y grande suele tardar más en adoptarse que una vieja y pequeña
nuevas_features['Age_x_MaturitySize'] = df_train['Age'] * df_train['MaturitySize']
nuevas_features['Age_x_MaturitySize'] = df_test['Age'] * df_test['MaturitySize']


# 2. Interacciones de Costo y Raza
# El impacto del Fee (costo) puede variar según si la raza es pura (Breed2 == 0) o no
nuevas_features['Fee_x_Breed1'] = df_train['Fee'] * df_train['Breed1']
nuevas_features['Fee_x_Breed1'] = df_test['Fee'] * df_test['Breed1']


# 3. Interacciones de Esfuerzo del Rescatista (Las más importantes según tu lista)
# ¿El rescatista que tiene muchas fotos (PhotoAmt) tiene mejor suerte con ciertas edades?
nuevas_features['Age_x_PhotoAmt'] = df_train['Age'] * df_train['PhotoAmt']
nuevas_features['Age_x_PhotoAmt'] = df_test['Age'] * df_test['PhotoAmt']

# 4. Interacciones de Salud y Cuidados
# Estar vacunado, desparasitado y esterilizado combinado
nuevas_features['Salud_Completa'] = df_train['Vaccinated'] * df_train['Dewormed'] * df_train['Sterilized']
nuevas_features['Salud_Completa'] = df_test['Vaccinated'] * df_test['Dewormed'] * df_test['Sterilized']

# 5. Ratio de Fotos por Cantidad de Mascotas
# Si hay muchos animales en la publicación (Quantity), ¿hay suficientes fotos para todos?
nuevas_features['Photos_per_Pet'] = df_train['PhotoAmt'] / (df_train['Quantity'] + 1)
nuevas_features['Photos_per_Pet'] = df_test['PhotoAmt'] / (df_test['Quantity'] + 1)

# 6. Cruces de Color y Raza (Estética)
nuevas_features['Breed_x_Color1'] = df_train['Breed1'] * df_train['Color1']
nuevas_features['Breed_x_Color1'] = df_test['Breed1'] * df_test['Color1']

# --- CONCATENACIÓN FINAL ---
print("Agregando interacciones seleccionadas al dataset...")
df_interacciones = pd.DataFrame(nuevas_features)
df_train = pd.concat([df_train, df_interacciones], axis=1)
df_test = pd.concat([df_test, df_interacciones], axis=1)

print(f"Se han agregado {len(nuevas_features)} columnas estratégicas.")
print(f"Nuevo tamaño del dataset: {df_train.shape}")

Agregando interacciones seleccionadas al dataset...
Se han agregado 6 columnas estratégicas.
Nuevo tamaño del dataset: (11994, 31)


In [92]:
def crear_variables_agregadas(df_train):
    # --- 1. Estadísticas por RescuerID (El perfil del rescatista) ---
    # Cantidad de mascotas subidas por cada rescatista
    df_train['rescu_total_pets'] = df_train.groupby('RescuerID')['RescuerID'].transform('count')
    
    # Promedio de fotos que sube este rescatista (indica calidad de perfil)
    df_train['rescu_avg_photos'] = df_train.groupby('RescuerID')['PhotoAmt'].transform('mean')
    
    # Promedio de precio que cobra este rescatista
    df_train['rescu_avg_fee'] = df_train.groupby('RescuerID')['Fee'].transform('mean')

    # --- 2. Estadísticas por Breed1 (El perfil de la raza) ---
    # ¿Qué tan común es esta raza en el dataset?
    df_train['breed_popularity'] = df_train.groupby('Breed1')['Breed1'].transform('count')
    
    # Promedio de edad para esta raza (algunas razas se abandonan más adultas)
    df_train['breed_avg_age'] = df_train.groupby('Breed1')['Age'].transform('mean')
    
    # Promedio de Fee para esta raza (hay razas que siempre son caras)
    df_train['breed_avg_fee'] = df_train.groupby('Breed1')['Fee'].transform('mean')

    # --- 3. Interacción: Rescatista x Raza ---
    # ¿Cuántas veces este rescatista trabajó con esta raza específica?
    df_train['rescu_breed_experience'] = df_train.groupby(['RescuerID', 'Breed1'])['PetID'].transform('count')

    return df_train

# Aplicamos a tu dataset
df_train = crear_variables_agregadas(df_train)
# Repetir lo mismo para el test_df si lo tenés aparte

def crear_variables_agregadas(df_test):
    # --- 1. Estadísticas por RescuerID (El perfil del rescatista) ---
    # Cantidad de mascotas subidas por cada rescatista
    df_test['rescu_total_pets'] = df_test.groupby('RescuerID')['RescuerID'].transform('count')
    
    # Promedio de fotos que sube este rescatista (indica calidad de perfil)
    df_test['rescu_avg_photos'] = df_test.groupby('RescuerID')['PhotoAmt'].transform('mean')
    
    # Promedio de precio que cobra este rescatista
    df_test['rescu_avg_fee'] = df_test.groupby('RescuerID')['Fee'].transform('mean')

    # --- 2. Estadísticas por Breed1 (El perfil de la raza) ---
    # ¿Qué tan común es esta raza en el dataset?
    df_test['breed_popularity'] = df_test.groupby('Breed1')['Breed1'].transform('count')
    
    # Promedio de edad para esta raza (algunas razas se abandonan más adultas)
    df_test['breed_avg_age'] = df_test.groupby('Breed1')['Age'].transform('mean')
    
    # Promedio de Fee para esta raza (hay razas que siempre son caras)
    df_test['breed_avg_fee'] = df_test.groupby('Breed1')['Fee'].transform('mean')

    # --- 3. Interacción: Rescatista x Raza ---
    # ¿Cuántas veces este rescatista trabajó con esta raza específica?
    df_test['rescu_breed_experience'] = df_test.groupby(['RescuerID', 'Breed1'])['PetID'].transform('count')

    return df_test

# Aplicamos a tu dataset
df_test = crear_variables_agregadas(df_test)

In [93]:
#Genero dataframes de train y test con sus respectivos targets
X_train = df_train.drop(columns=['AdoptionSpeed'])
y_train = df_train['AdoptionSpeed']

X_test = df_test.drop(columns=['AdoptionSpeed'])
y_test = df_test['AdoptionSpeed']

In [96]:
# 1. Definir explícitamente qué variables son categóricas
categorical_cols = ['Type', 'Breed1', 'Breed2', 'Gender', 'Color1', 'Color2', 'Color3', 
                    'Vaccinated', 'Dewormed', 'Sterilized', 'Health', 'State','SizeCategory','Name','PetID','Description','RescuerID']


# Asegurarse de que el modelo entienda que son categorías (convertir a tipo 'category' en pandas)
for col in categorical_cols:
    X_train[col] = X_train[col].astype('category')
    X_test[col] = X_test[col].astype('category')

In [97]:
X_train

,Type,Name,Age,Breed1,Breed2,Gender,Color1,Color2,Color3,MaturitySize,...,Salud_Completa,Photos_per_Pet,Breed_x_Color1,rescu_total_pets,rescu_avg_photos,rescu_avg_fee,breed_popularity,breed_avg_age,breed_avg_fee,rescu_breed_experience
0,1,The Adorable Trio,2,307,307,1,1,0,0,2,...,8.0,3.000000,307.0,1,8.000000,0.000000,4754,7.558687,9.696466,1
1,1,Perky,12,307,0,2,2,0,0,1,...,2.0,1.000000,103.0,33,2.181818,0.000000,4754,7.558687,9.696466,12
2,1,Bernard Boy,2,307,307,1,1,2,7,2,...,2.0,0.833333,614.0,31,5.193548,52.870968,4754,7.558687,9.696466,28
3,1,Oreo 1,2,307,307,1,3,0,0,2,...,2.0,3.000000,530.0,180,3.127778,11.111111,4754,7.558687,9.696466,165
4,2,Anak Kucing,4,265,0,3,1,2,0,2,...,8.0,2.666667,532.0,1,2.000000,0.000000,983,6.342828,10.769074,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11989,2,Maxine,8,265,299,2,1,2,0,2,...,NaN,NaN,NaN,1,2.000000,0.000000,983,6.342828,10.769074,1
11990,1,Browny,2,307,0,2,6,7,0,1,...,NaN,NaN,NaN,1,1.000000,0.000000,4754,7.558687,9.696466,1
11991,1,Midnight,2,307,307,1,1,0,0,2,...,NaN,NaN,NaN,4,2.000000,0.000000,4754,7.558687,9.696466,4
11992,1,Boxer,2,307,307,1,1,2,7,2,...,NaN,NaN,NaN,111,2.234234,0.000000,4754,7.558687,9.696466,99


In [30]:
import lightgbm as lgb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# ============================================================================
# 1. Preparación de Datasets (Uso de validación real)
# ============================================================================
# Es fundamental que valid_sets reciba datos que el modelo NO vio en el training
dtrain = lgb.Dataset(X_train, label=y_train, free_raw_data=False)
dvalid = lgb.Dataset(X_test, label=y_test, reference=dtrain)

# ============================================================================
# 2. Parámetros y Entrenamiento
# ============================================================================
params = {
    'objective': 'multiclass',
    'num_class': 5,
    'metric': 'multi_error',
    'verbosity': -1,
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': 7,
    'min_data_in_leaf': 20,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'random_state': 42
}

print("Entrenando modelo con Early Stopping...")
model = lgb.train(
    params,
    dtrain,
    num_boost_round=1000,
    valid_sets=[dtrain, dvalid],
    valid_names=['train', 'valid'],
    callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=True)]
)

# ============================================================================
# 3. Importancia de Variables (Gain)
# ============================================================================
importance_type = 'gain' 

feature_importances = model.feature_importance(importance_type=importance_type)
feature_names = X_train.columns

# Crear DataFrame de importancia
df_importance = pd.DataFrame({
    'feature': feature_names,
    'importance': feature_importances,
    'importance_pct': (feature_importances / feature_importances.sum() * 100).round(2) if feature_importances.sum() > 0 else 0
}).sort_values('importance', ascending=False).reset_index(drop=True)

# Mostrar resultados
print("\n=== Top 60 Importancia de Features (Gain) ===")
print(df_importance.head(60))


Entrenando modelo con Early Stopping...
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[31]	train's multi_error: 0.52618	valid's multi_error: 0.614538

=== Top 60 Importancia de Features (Gain) ===
         feature   importance  importance_pct
0            Age  9069.527051           21.71
1         Breed1  7748.633011           18.55
2       PhotoAmt  5801.997793           13.89
3     Sterilized  3597.491404            8.61
4       Quantity  2063.976989            4.94
5            Fee  1412.836911            3.38
6   MaturitySize  1320.117599            3.16
7          State  1305.514130            3.12
8         Breed2  1279.608050            3.06
9         Gender  1247.400281            2.99
10      Dewormed  1229.186012            2.94
11    Vaccinated  1183.843136            2.83
12     FurLength   912.608391            2.18
13          Type   863.749701            2.07
14          Name   819.123665            1.96
15        Color1 

In [99]:
import pandas as pd
import numpy as np
from sklearn.metrics import average_precision_score

# ==========================================================
# 1. CÁLCULO DE GAIN (TABLA 1)
# ==========================================================
print("Calculando importancia por Gain...")
df_gain = pd.DataFrame({
    'feature': X_train.columns,
    'gain': model.feature_importance(importance_type='gain')
})

# ==========================================================
# 2. CÁLCULO DE PERMUTATION IMPACT (TABLA 2)
# ==========================================================
print("\nIniciando auditoría de impacto en Test (Permutation)...")

def get_full_audit(model, X_test_audit, y):
    # Aseguramos que el orden de columnas sea idéntico al de entrenamiento
    features = X_train.columns.tolist()
    X_test_audit = X_test_audit[features]
    
    # Predicción base
    y_pred_base = model.predict(X_test_audit).reshape(-1, 1)
    baseline_score = average_precision_score(y, y_pred_base)
    
    print(f"Baseline PR-AUC en Test: {baseline_score:.4f}")
    
    results = []
    
    for i, col in enumerate(features):
        # Copiamos la columna original para no romper el DataFrame
        original_col = X_test_audit[col].copy()
        
        # PERMUTACIÓN:
        # Si es categórica, debemos mantener el tipo 'category' al permutar
        if X_test_audit[col].dtype.name == 'category':
            permuted_values = np.random.permutation(X_test_audit[col].values)
            X_test_audit[col] = pd.Series(permuted_values).astype('category').values
            # Reasignamos las categorías originales para que LightGBM no se queje
            X_test_audit[col] = X_test_audit[col].cat.set_categories(original_col.cat.categories)
        else:
            X_test_audit[col] = np.random.permutation(X_test_audit[col].values)
        
        # Medimos impacto con el modelo
        try:
            y_pred_shuffled = model.predict(X_test_audit).reshape(-1, 1)
            shuffled_score = average_precision_score(y, y_pred_shuffled)
            drop = baseline_score - shuffled_score
        except ValueError:
            # Si falla por categorías, forzamos el re-tipado rápido
            for c in X_test_audit.select_dtypes(['category']).columns:
                X_test_audit[c] = X_test_audit[c].astype('category')
            y_pred_shuffled = model.predict(X_test_audit).reshape(-1, 1)
            shuffled_score = average_precision_score(y, y_pred_shuffled)
            drop = baseline_score - shuffled_score

        results.append(drop)
        
        # RESTAURACIÓN: fundamental devolver la columna a su estado previo
        X_test_audit[col] = original_col
        
        if (i+1) % 5 == 0 or i == 0:
            print(f"[{i+1}/{len(features)}] Auditada: {col} | Drop: {drop:.6f}")
            
    return pd.DataFrame({'feature': features, 'test_impact': results})

# Ejecutamos la auditoría sobre una copia limpia
df_audit = get_full_audit(model, X_test.copy(), y_test)

# ==========================================================
# 3. UNIFICACIÓN Y TABLA FINAL
# ==========================================================
df_final = pd.merge(df_gain, df_audit, on='feature')

# Ordenamos por impacto real en Test (Permutation)
df_final = df_final.sort_values('test_impact', ascending=False).reset_index(drop=True)

# Agregamos % de Gain para comparar
df_final['gain_pct'] = (df_final['gain'] / df_final['gain'].sum() * 100).round(2)

print("\n" + "="*60)
print("=== MATRIZ DE DECISIÓN FINAL (GAIN vs IMPACT) ===")
print("="*60)
print(df_final[['feature', 'gain', 'test_impact', 'gain_pct']].to_string(index=False))

# Guardar
df_final.to_csv('matriz_final_maestria.csv', index=False)

Calculando importancia por Gain...

Iniciando auditoría de impacto en Test (Permutation)...
Baseline PR-AUC en Test: 0.0186
[1/37] Auditada: Type | Drop: -0.000000
[5/37] Auditada: Breed2 | Drop: -0.000341
[10/37] Auditada: MaturitySize | Drop: 0.000064
[15/37] Auditada: Health | Drop: 0.000019
[20/37] Auditada: VideoAmt | Drop: 0.000006
[25/37] Auditada: Age_x_MaturitySize | Drop: -0.000100
[30/37] Auditada: Breed_x_Color1 | Drop: 0.000160
[35/37] Auditada: breed_avg_age | Drop: -0.000057

=== MATRIZ DE DECISIÓN FINAL (GAIN vs IMPACT) ===
               feature         gain   test_impact  gain_pct
                Breed1  9231.193295  3.342429e-04      3.94
                   Age 10851.749149  1.830042e-04      4.63
        Breed_x_Color1  2395.813111  1.600452e-04      1.02
              Dewormed  4276.702014  1.218434e-04      1.82
        Salud_Completa  2423.510235  1.152169e-04      1.03
                Color1  2375.482496  1.063340e-04      1.01
          MaturitySize  2730.74150

In [12]:
import optuna
import lightgbm as lgb
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score

# Asegúrate de tener X_train, y_train y categorical_cols definidos previamente

def objective(trial):
    # 1. Definir el espacio de búsqueda de hiperparámetros
    param = {
        'objective': 'multiclass',
        'num_class': 5,
        'metric': 'multi_error',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'seed': 42,

        # Hiperparámetros a optimizar por Optuna
        'learning_rate': trial.suggest_float('learning_rate', 0.05, 0.5, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 50, 200),
        'max_depth': trial.suggest_int('max_depth', 7, 50),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 10, 50),
        'min_child_samples': trial.suggest_int('min_child_samples', 100, 3000),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 40, 120, 1),
        'lambda_l1': trial.suggest_float('lambda_l1',0.001,0.5, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2',0.01,5.0, log=True),
        'min_gain_to_split':    trial.suggest_float('min_gain_to_split', 0.01, 0.2, log=True), # Peaje más caro para abrir ramas 
    }


    # 2. Configurar la Validación Cruzada
    N_SPLITS = 5
    folds = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
    oof_preds = np.zeros((len(X_train), 5))

    # 3. Bucle de entrenamiento por Fold
    for fold_, (trn_idx, val_idx) in enumerate(folds.split(X_train, y_train)):
        X_tr, y_tr = X_train.iloc[trn_idx], y_train.iloc[trn_idx]
        X_val, y_val = X_train.iloc[val_idx], y_train.iloc[val_idx]
        
        trn_data = lgb.Dataset(X_tr, label=y_tr, categorical_feature=categorical_cols)
        val_data = lgb.Dataset(X_val, label=y_val, categorical_feature=categorical_cols, reference=trn_data)
        
        # Entrenar el modelo
        clf = lgb.train(
            param,
            trn_data,
            num_boost_round=500, # Un número alto, early_stopping lo detendrá antes
            valid_sets=[trn_data, val_data],
            callbacks=[
                lgb.early_stopping(stopping_rounds=30, verbose=False)
            ]
        )
        
        # Guardar predicciones de validación
        oof_preds[val_idx] = clf.predict(X_val, num_iteration=clf.best_iteration)

    # 4. Calcular la métrica final (QWK) para este Trial
    oof_predictions_classes = oof_preds.argmax(axis=1)
    kappa_score = cohen_kappa_score(y_train, oof_predictions_classes, weights='quadratic')
    
    return kappa_score

# --- EJECUCIÓN DE LA OPTIMIZACIÓN ---

print("Iniciando búsqueda de hiperparámetros con Optuna...")

# Queremos MAXIMIZAR el score Kappa
study = optuna.create_study(direction='maximize', study_name="Petfinder_LGBM_Optuna")

# Ejecutar el estudio (puedes cambiar n_trials dependiendo del tiempo que tengas)
study.optimize(objective, n_trials=500)

# Mostrar resultados
print("\n--- ¡Optimización Terminada! ---")
print(f"Mejor Score Kappa (OOF): {study.best_value:.4f}")
print("Mejores Hiperparámetros encontrados:")
for key, value in study.best_params.items():
    print(f"    '{key}': {value}")

c:\Users\fchiavassa\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-04-09 19:46:11,919] A new study created in memory with name: Petfinder_LGBM_Optuna
C:\Users\fchiavassa\AppData\Local\Temp\ipykernel_23656\1294875881.py:27: FutureWarning: suggest_int() got {'step'} as positional arguments but they were expected to be given as keyword arguments.
Positional arguments ['self', 'name', 'low', 'high', 'step', 'log'] in suggest_int() have been deprecated since v3.5.0. They will be replaced with the corresponding keyword arguments in v5.0.0, so please use the keyword specification instead. See https://github.com/optuna/optuna/releases/tag/v3.5.0 for details.
  'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 40, 120, 1),


Iniciando búsqueda de hiperparámetros con Optuna...


[I 2026-04-09 19:46:18,437] Trial 0 finished with value: 0.3824249379001148 and parameters: {'learning_rate': 0.35201421741381955, 'num_leaves': 94, 'max_depth': 17, 'feature_fraction': 0.9128877181712357, 'bagging_fraction': 0.9834534507546357, 'bagging_freq': 48, 'min_child_samples': 509, 'min_data_in_leaf': 83, 'lambda_l1': 0.0015504433049905945, 'lambda_l2': 0.3547821134921013, 'min_gain_to_split': 0.02126456081939171}. Best is trial 0 with value: 0.3824249379001148.
C:\Users\fchiavassa\AppData\Local\Temp\ipykernel_23656\1294875881.py:27: FutureWarning: suggest_int() got {'step'} as positional arguments but they were expected to be given as keyword arguments.
Positional arguments ['self', 'name', 'low', 'high', 'step', 'log'] in suggest_int() have been deprecated since v3.5.0. They will be replaced with the corresponding keyword arguments in v5.0.0, so please use the keyword specification instead. See https://github.com/optuna/optuna/releases/tag/v3.5.0 for details.
  'min_data_in_

KeyboardInterrupt: 

In [102]:
import optuna
import lightgbm as lgb
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score

# Asegúrate de tener X_train, y_train y categorical_cols definidos previamente

def objective(trial):
    # 1. Definir el espacio de búsqueda de hiperparámetros
    param = {
        'objective': 'multiclass',
        'num_class': 5,
        'metric': 'multi_error',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'seed': 42,

        # Hiperparámetros a optimizar por Optuna
        'learning_rate': trial.suggest_float('learning_rate', 0.03, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 100, 250),
        'max_depth': trial.suggest_int('max_depth', 22, 60),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.7, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.7, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 15, 30),
        'min_child_samples': trial.suggest_int('min_child_samples', 1000, 3000),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 40, 100, 1),
        'lambda_l1': trial.suggest_float('lambda_l1',0.001,0.1, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2',0.01,1.0, log=True),
        'min_gain_to_split':    trial.suggest_float('min_gain_to_split', 0.01, 0.1, log=True), # Peaje más caro para abrir ramas 
    }

    # 2. Configurar la Validación Cruzada
    N_SPLITS = 5
    folds = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
    oof_preds = np.zeros((len(X_train), 5))

    # 3. Bucle de entrenamiento por Fold
    for fold_, (trn_idx, val_idx) in enumerate(folds.split(X_train, y_train)):
        X_tr, y_tr = X_train.iloc[trn_idx], y_train.iloc[trn_idx]
        X_val, y_val = X_train.iloc[val_idx], y_train.iloc[val_idx]
        
        trn_data = lgb.Dataset(X_tr, label=y_tr, categorical_feature=categorical_cols)
        val_data = lgb.Dataset(X_val, label=y_val, categorical_feature=categorical_cols, reference=trn_data)
        
        # Entrenar el modelo
        clf = lgb.train(
            param,
            trn_data,
            num_boost_round=500, # Un número alto, early_stopping lo detendrá antes
            valid_sets=[trn_data, val_data],
            callbacks=[
                lgb.early_stopping(stopping_rounds=30, verbose=False)
            ]
        )
        
        # Guardar predicciones de validación
        oof_preds[val_idx] = clf.predict(X_val, num_iteration=clf.best_iteration)

    # 4. Calcular la métrica final (QWK) para este Trial
    oof_predictions_classes = oof_preds.argmax(axis=1)
    kappa_score = cohen_kappa_score(y_train, oof_predictions_classes, weights='quadratic')
    
    return kappa_score

# --- EJECUCIÓN DE LA OPTIMIZACIÓN ---

print("Iniciando búsqueda de hiperparámetros con Optuna...")

# Queremos MAXIMIZAR el score Kappa
study = optuna.create_study(direction='maximize', study_name="Petfinder_LGBM_Optuna")

# Ejecutar el estudio (puedes cambiar n_trials dependiendo del tiempo que tengas)
study.optimize(objective, n_trials=500)

# Mostrar resultados
print("\n--- ¡Optimización Terminada! ---")
print(f"Mejor Score Kappa (OOF): {study.best_value:.4f}")
print("Mejores Hiperparámetros encontrados:")
for key, value in study.best_params.items():
    print(f"    '{key}': {value}")

[I 2026-04-08 12:50:07,264] A new study created in memory with name: Petfinder_LGBM_Optuna


Iniciando búsqueda de hiperparámetros con Optuna...


C:\Users\fchiavassa\AppData\Local\Temp\ipykernel_44164\2282963097.py:27: FutureWarning: suggest_int() got {'step'} as positional arguments but they were expected to be given as keyword arguments.
Positional arguments ['self', 'name', 'low', 'high', 'step', 'log'] in suggest_int() have been deprecated since v3.5.0. They will be replaced with the corresponding keyword arguments in v5.0.0, so please use the keyword specification instead. See https://github.com/optuna/optuna/releases/tag/v3.5.0 for details.
  'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 40, 100, 1),
[I 2026-04-08 12:50:19,322] Trial 0 finished with value: 0.40662448046712507 and parameters: {'learning_rate': 0.04448103549624654, 'num_leaves': 212, 'max_depth': 45, 'feature_fraction': 0.9321236252962755, 'bagging_fraction': 0.952525001997167, 'bagging_freq': 15, 'min_child_samples': 2268, 'min_data_in_leaf': 67, 'lambda_l1': 0.05554702068756175, 'lambda_l2': 0.03393596022318814, 'min_gain_to_split': 0.024669446


--- ¡Optimización Terminada! ---
Mejor Score Kappa (OOF): 0.4141
Mejores Hiperparámetros encontrados:
    'learning_rate': 0.06979818452348317
    'num_leaves': 233
    'max_depth': 60
    'feature_fraction': 0.9165132580606841
    'bagging_fraction': 0.9892359436440317
    'bagging_freq': 18
    'min_child_samples': 1531
    'min_data_in_leaf': 47
    'lambda_l1': 0.011349471900716017
    'lambda_l2': 0.013799677385552779
    'min_gain_to_split': 0.010447860690942043


In [105]:
import optuna
import lightgbm as lgb
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score

# Asegúrate de tener X_train, y_train y categorical_cols definidos previamente

def objective(trial):
    # 1. Definir el espacio de búsqueda de hiperparámetros
    param = {
        'objective': 'multiclass',
        'num_class': 5,
        'metric': 'multi_error',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'seed': 42,

        # Hiperparámetros a optimizar por Optuna
        'learning_rate': trial.suggest_float('learning_rate', 0.05, 0.8, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 200, 270),
        'max_depth': trial.suggest_int('max_depth', 50, 80),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.85, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.85, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 15, 25),
        'min_child_samples': trial.suggest_int('min_child_samples', 1000, 2000),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 40, 60, 1),
        'lambda_l1': trial.suggest_float('lambda_l1',0.001,0.025, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2',0.005,0.05, log=True),
        'min_gain_to_split':    trial.suggest_float('min_gain_to_split', 0.07, 0.15, log=True), # Peaje más caro para abrir ramas 
    }
    # 2. Configurar la Validación Cruzada
    N_SPLITS = 5
    folds = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
    oof_preds = np.zeros((len(X_train), 5))

    # 3. Bucle de entrenamiento por Fold
    for fold_, (trn_idx, val_idx) in enumerate(folds.split(X_train, y_train)):
        X_tr, y_tr = X_train.iloc[trn_idx], y_train.iloc[trn_idx]
        X_val, y_val = X_train.iloc[val_idx], y_train.iloc[val_idx]
        
        trn_data = lgb.Dataset(X_tr, label=y_tr, categorical_feature=categorical_cols)
        val_data = lgb.Dataset(X_val, label=y_val, categorical_feature=categorical_cols, reference=trn_data)
        
        # Entrenar el modelo
        clf = lgb.train(
            param,
            trn_data,
            num_boost_round=500, # Un número alto, early_stopping lo detendrá antes
            valid_sets=[trn_data, val_data],
            callbacks=[
                lgb.early_stopping(stopping_rounds=30, verbose=False)
            ]
        )
        
        # Guardar predicciones de validación
        oof_preds[val_idx] = clf.predict(X_val, num_iteration=clf.best_iteration)

    # 4. Calcular la métrica final (QWK) para este Trial
    oof_predictions_classes = oof_preds.argmax(axis=1)
    kappa_score = cohen_kappa_score(y_train, oof_predictions_classes, weights='quadratic')
    
    return kappa_score

# --- EJECUCIÓN DE LA OPTIMIZACIÓN ---

print("Iniciando búsqueda de hiperparámetros con Optuna...")

# Queremos MAXIMIZAR el score Kappa
study = optuna.create_study(direction='maximize', study_name="Petfinder_LGBM_Optuna")

# Ejecutar el estudio (puedes cambiar n_trials dependiendo del tiempo que tengas)
study.optimize(objective, n_trials=1000)

# Mostrar resultados
print("\n--- ¡Optimización Terminada! ---")
print(f"Mejor Score Kappa (OOF): {study.best_value:.4f}")
print("Mejores Hiperparámetros encontrados:")
for key, value in study.best_params.items():
    print(f"    '{key}': {value}")
    

[I 2026-04-08 23:39:31,606] A new study created in memory with name: Petfinder_LGBM_Optuna


Iniciando búsqueda de hiperparámetros con Optuna...


C:\Users\fchiavassa\AppData\Local\Temp\ipykernel_44164\3606145962.py:27: FutureWarning: suggest_int() got {'step'} as positional arguments but they were expected to be given as keyword arguments.
Positional arguments ['self', 'name', 'low', 'high', 'step', 'log'] in suggest_int() have been deprecated since v3.5.0. They will be replaced with the corresponding keyword arguments in v5.0.0, so please use the keyword specification instead. See https://github.com/optuna/optuna/releases/tag/v3.5.0 for details.
  'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 40, 60, 1),
[I 2026-04-08 23:39:45,568] Trial 0 finished with value: 0.39902299647791395 and parameters: {'learning_rate': 0.0711488277092433, 'num_leaves': 218, 'max_depth': 62, 'feature_fraction': 0.905214937314951, 'bagging_fraction': 0.9992312517553792, 'bagging_freq': 16, 'min_child_samples': 1919, 'min_data_in_leaf': 60, 'lambda_l1': 0.011561735792353929, 'lambda_l2': 0.02109902502736927, 'min_gain_to_split': 0.1472724534


--- ¡Optimización Terminada! ---
Mejor Score Kappa (OOF): 0.4147
Mejores Hiperparámetros encontrados:
    'learning_rate': 0.10757105093289668
    'num_leaves': 213
    'max_depth': 76
    'feature_fraction': 0.8844720212991225
    'bagging_fraction': 0.9248376103423376
    'bagging_freq': 23
    'min_child_samples': 1029
    'min_data_in_leaf': 58
    'lambda_l1': 0.0010129469093082858
    'lambda_l2': 0.02128555488699726
    'min_gain_to_split': 0.09951613060656529


In [106]:
import optuna
import lightgbm as lgb
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score

# Asegúrate de tener X_train, y_train y categorical_cols definidos previamente

def objective(trial):
    # 1. Definir el espacio de búsqueda de hiperparámetros
    param = {
        'objective': 'multiclass',
        'num_class': 5,
        'metric': 'multi_error',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'seed': 42,

        # Hiperparámetros a optimizar por Optuna
        'learning_rate': trial.suggest_float('learning_rate', 0.05, 0.2, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 200, 240),
        'max_depth': trial.suggest_int('max_depth', 35, 80),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.86, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.86, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 17, 24),
        'min_child_samples': trial.suggest_int('min_child_samples', 1000, 1800, log=True),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 45, 60, 1),
        'lambda_l1': trial.suggest_float('lambda_l1',0.001,0.01, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2',0.005,0.025, log=True),
        'min_gain_to_split':    trial.suggest_float('min_gain_to_split', 0.07, 0.13, log=True), # Peaje más caro para abrir ramas 
    }

'learning_rate': 0.06944835999597897, 
'num_leaves': 213, 
'max_depth': 37, 
'feature_fraction': 0.872345644337963, 
'bagging_fraction': 0.965774173133069, 
'bagging_freq': 19, 
'min_child_samples': 1298, 
'min_data_in_leaf': 57, 
'lambda_l1': 0.006012766850092706, 
'lambda_l2': 0.008100955457339194, 
'min_gain_to_split': 0.07198419292196633

    # 2. Configurar la Validación Cruzada
    N_SPLITS = 5
    folds = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
    oof_preds = np.zeros((len(X_train), 5))

    # 3. Bucle de entrenamiento por Fold
    for fold_, (trn_idx, val_idx) in enumerate(folds.split(X_train, y_train)):
        X_tr, y_tr = X_train.iloc[trn_idx], y_train.iloc[trn_idx]
        X_val, y_val = X_train.iloc[val_idx], y_train.iloc[val_idx]
        
        trn_data = lgb.Dataset(X_tr, label=y_tr, categorical_feature=categorical_cols)
        val_data = lgb.Dataset(X_val, label=y_val, categorical_feature=categorical_cols, reference=trn_data)
        
        # Entrenar el modelo
        clf = lgb.train(
            param,
            trn_data,
            num_boost_round=500, # Un número alto, early_stopping lo detendrá antes
            valid_sets=[trn_data, val_data],
            callbacks=[
                lgb.early_stopping(stopping_rounds=30, verbose=False)
            ]
        )
        
        # Guardar predicciones de validación
        oof_preds[val_idx] = clf.predict(X_val, num_iteration=clf.best_iteration)

    # 4. Calcular la métrica final (QWK) para este Trial
    oof_predictions_classes = oof_preds.argmax(axis=1)
    kappa_score = cohen_kappa_score(y_train, oof_predictions_classes, weights='quadratic')
    
    return kappa_score

# --- EJECUCIÓN DE LA OPTIMIZACIÓN ---

print("Iniciando búsqueda de hiperparámetros con Optuna...")

# Queremos MAXIMIZAR el score Kappa
study = optuna.create_study(direction='maximize', study_name="Petfinder_LGBM_Optuna")

# Ejecutar el estudio (puedes cambiar n_trials dependiendo del tiempo que tengas)
study.optimize(objective, n_trials=500)

# Mostrar resultados
print("\n--- ¡Optimización Terminada! ---")
print(f"Mejor Score Kappa (OOF): {study.best_value:.4f}")
print("Mejores Hiperparámetros encontrados:")
for key, value in study.best_params.items():
    print(f"    '{key}': {value}")
    

[I 2026-04-09 08:22:46,221] A new study created in memory with name: Petfinder_LGBM_Optuna


Iniciando búsqueda de hiperparámetros con Optuna...


C:\Users\fchiavassa\AppData\Local\Temp\ipykernel_44164\1284922978.py:27: FutureWarning: suggest_int() got {'step'} as positional arguments but they were expected to be given as keyword arguments.
Positional arguments ['self', 'name', 'low', 'high', 'step', 'log'] in suggest_int() have been deprecated since v3.5.0. They will be replaced with the corresponding keyword arguments in v5.0.0, so please use the keyword specification instead. See https://github.com/optuna/optuna/releases/tag/v3.5.0 for details.
  'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 45, 60, 1),
[I 2026-04-09 08:22:54,496] Trial 0 finished with value: 0.3973338303173921 and parameters: {'learning_rate': 0.07528814817438165, 'num_leaves': 233, 'max_depth': 40, 'feature_fraction': 0.8899962563322092, 'bagging_fraction': 0.9994906452688412, 'bagging_freq': 23, 'min_child_samples': 1164, 'min_data_in_leaf': 45, 'lambda_l1': 0.0037561442631910127, 'lambda_l2': 0.02348513508304213, 'min_gain_to_split': 0.10203313


--- ¡Optimización Terminada! ---
Mejor Score Kappa (OOF): 0.4154
Mejores Hiperparámetros encontrados:
    'learning_rate': 0.06944835999597897
    'num_leaves': 213
    'max_depth': 37
    'feature_fraction': 0.872345644337963
    'bagging_fraction': 0.965774173133069
    'bagging_freq': 19
    'min_child_samples': 1298
    'min_data_in_leaf': 57
    'lambda_l1': 0.006012766850092706
    'lambda_l2': 0.008100955457339194
    'min_gain_to_split': 0.07198419292196633


In [ ]:
import optuna
import lightgbm as lgb
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score

# Asegúrate de tener X_train, y_train y categorical_cols definidos previamente

def objective(trial):
    # 1. Definir el espacio de búsqueda de hiperparámetros
    param = {
        'objective': 'multiclass',
        'num_class': 5,
        'metric': 'multi_error',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'seed': 42,

        # Hiperparámetros a optimizar por Optuna
        'learning_rate': trial.suggest_float('learning_rate', 0.05, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 200, 230),
        'max_depth': trial.suggest_int('max_depth', 35, 45),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.86, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.95, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 17, 24),
        'min_child_samples': trial.suggest_int('min_child_samples', 1000, 1500, log=True),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 51, 61, 1),
        'lambda_l1': trial.suggest_float('lambda_l1',0.001,0.005, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2',0.005,0.012, log=True),
        'min_gain_to_split':    trial.suggest_float('min_gain_to_split', 0.06, 0.1, log=True), # Peaje más caro para abrir ramas 
    }

    # 2. Configurar la Validación Cruzada
    N_SPLITS = 5
    folds = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
    oof_preds = np.zeros((len(X_train), 5))

    # 3. Bucle de entrenamiento por Fold
    for fold_, (trn_idx, val_idx) in enumerate(folds.split(X_train, y_train)):
        X_tr, y_tr = X_train.iloc[trn_idx], y_train.iloc[trn_idx]
        X_val, y_val = X_train.iloc[val_idx], y_train.iloc[val_idx]
        
        trn_data = lgb.Dataset(X_tr, label=y_tr, categorical_feature=categorical_cols)
        val_data = lgb.Dataset(X_val, label=y_val, categorical_feature=categorical_cols, reference=trn_data)
        
        # Entrenar el modelo
        clf = lgb.train(
            param,
            trn_data,
            num_boost_round=500, # Un número alto, early_stopping lo detendrá antes
            valid_sets=[trn_data, val_data],
            callbacks=[
                lgb.early_stopping(stopping_rounds=30, verbose=False)
            ]
        )
        
        # Guardar predicciones de validación
        oof_preds[val_idx] = clf.predict(X_val, num_iteration=clf.best_iteration)

    # 4. Calcular la métrica final (QWK) para este Trial
    oof_predictions_classes = oof_preds.argmax(axis=1)
    kappa_score = cohen_kappa_score(y_train, oof_predictions_classes, weights='quadratic')
    
    return kappa_score

# --- EJECUCIÓN DE LA OPTIMIZACIÓN ---

print("Iniciando búsqueda de hiperparámetros con Optuna...")

# Queremos MAXIMIZAR el score Kappa
study = optuna.create_study(direction='maximize', study_name="Petfinder_LGBM_Optuna")

# Ejecutar el estudio (puedes cambiar n_trials dependiendo del tiempo que tengas)
study.optimize(objective, n_trials=500)

# Mostrar resultados
print("\n--- ¡Optimización Terminada! ---")
print(f"Mejor Score Kappa (OOF): {study.best_value:.4f}")
print("Mejores Hiperparámetros encontrados:")
for key, value in study.best_params.items():
    print(f"    '{key}': {value}")
    

[I 2026-04-09 10:19:25,825] A new study created in memory with name: Petfinder_LGBM_Optuna


Iniciando búsqueda de hiperparámetros con Optuna...


C:\Users\fchiavassa\AppData\Local\Temp\ipykernel_44164\1601356527.py:27: FutureWarning: suggest_int() got {'step'} as positional arguments but they were expected to be given as keyword arguments.
Positional arguments ['self', 'name', 'low', 'high', 'step', 'log'] in suggest_int() have been deprecated since v3.5.0. They will be replaced with the corresponding keyword arguments in v5.0.0, so please use the keyword specification instead. See https://github.com/optuna/optuna/releases/tag/v3.5.0 for details.
  'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 51, 61, 1),
[I 2026-04-09 10:19:37,788] Trial 0 finished with value: 0.4024105536963525 and parameters: {'learning_rate': 0.054258391411151484, 'num_leaves': 220, 'max_depth': 39, 'feature_fraction': 0.9681054478654658, 'bagging_fraction': 0.9535983066268133, 'bagging_freq': 24, 'min_child_samples': 1252, 'min_data_in_leaf': 54, 'lambda_l1': 0.002866298021326141, 'lambda_l2': 0.009065459631746372, 'min_gain_to_split': 0.0668300


--- ¡Optimización Terminada! ---
Mejor Score Kappa (OOF): 0.4155
Mejores Hiperparámetros encontrados:
    'learning_rate': 0.08971210430879904
    'num_leaves': 209
    'max_depth': 36
    'feature_fraction': 0.8980577412306568
    'bagging_fraction': 0.9888816790376405
    'bagging_freq': 17
    'min_child_samples': 1010
    'min_data_in_leaf': 61
    'lambda_l1': 0.0020519474560541156
    'lambda_l2': 0.005919979899030505
    'min_gain_to_split': 0.06505635341081556


In [ ]:
import optuna
import lightgbm as lgb
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score

# Asegúrate de tener X_train, y_train y categorical_cols definidos previamente

def objective(trial):
    # 1. Definir el espacio de búsqueda de hiperparámetros
    param = {
        'objective': 'multiclass',
        'num_class': 5,
        'metric': 'multi_error',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'seed': 42,

        # Hiperparámetros a optimizar por Optuna
        'learning_rate': trial.suggest_float('learning_rate', 0.07, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 200, 230),
        'max_depth': trial.suggest_int('max_depth', 35, 40),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.86, 0.90),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.95, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 17, 20),
        'min_child_samples': trial.suggest_int('min_child_samples', 1000, 1100, log=True),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 60, 65, 1),
        'lambda_l1': trial.suggest_float('lambda_l1',0.001,0.003, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2',0.005,0.01, log=True),
        'min_gain_to_split':    trial.suggest_float('min_gain_to_split', 0.06, 0.08, log=True), # Peaje más caro para abrir ramas 
    }

    # 2. Configurar la Validación Cruzada
    N_SPLITS = 5
    folds = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
    oof_preds = np.zeros((len(X_train), 5))

    # 3. Bucle de entrenamiento por Fold
    for fold_, (trn_idx, val_idx) in enumerate(folds.split(X_train, y_train)):
        X_tr, y_tr = X_train.iloc[trn_idx], y_train.iloc[trn_idx]
        X_val, y_val = X_train.iloc[val_idx], y_train.iloc[val_idx]
        
        trn_data = lgb.Dataset(X_tr, label=y_tr, categorical_feature=categorical_cols)
        val_data = lgb.Dataset(X_val, label=y_val, categorical_feature=categorical_cols, reference=trn_data)
        
        # Entrenar el modelo
        clf = lgb.train(
            param,
            trn_data,
            num_boost_round=500, # Un número alto, early_stopping lo detendrá antes
            valid_sets=[trn_data, val_data],
            callbacks=[
                lgb.early_stopping(stopping_rounds=30, verbose=False)
            ]
        )
        
        # Guardar predicciones de validación
        oof_preds[val_idx] = clf.predict(X_val, num_iteration=clf.best_iteration)

    # 4. Calcular la métrica final (QWK) para este Trial
    oof_predictions_classes = oof_preds.argmax(axis=1)
    kappa_score = cohen_kappa_score(y_train, oof_predictions_classes, weights='quadratic')
    
    return kappa_score

# --- EJECUCIÓN DE LA OPTIMIZACIÓN ---

print("Iniciando búsqueda de hiperparámetros con Optuna...")

# Queremos MAXIMIZAR el score Kappa
study = optuna.create_study(direction='maximize', study_name="Petfinder_LGBM_Optuna")

# Ejecutar el estudio (puedes cambiar n_trials dependiendo del tiempo que tengas)
study.optimize(objective, n_trials=500)

# Mostrar resultados
print("\n--- ¡Optimización Terminada! ---")
print(f"Mejor Score Kappa (OOF): {study.best_value:.4f}")
print("Mejores Hiperparámetros encontrados:")
for key, value in study.best_params.items():
    print(f"    '{key}': {value}")
    

[I 2026-04-09 13:20:19,549] A new study created in memory with name: Petfinder_LGBM_Optuna


Iniciando búsqueda de hiperparámetros con Optuna...


C:\Users\fchiavassa\AppData\Local\Temp\ipykernel_44164\1189143466.py:27: FutureWarning: suggest_int() got {'step'} as positional arguments but they were expected to be given as keyword arguments.
Positional arguments ['self', 'name', 'low', 'high', 'step', 'log'] in suggest_int() have been deprecated since v3.5.0. They will be replaced with the corresponding keyword arguments in v5.0.0, so please use the keyword specification instead. See https://github.com/optuna/optuna/releases/tag/v3.5.0 for details.
  'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 60, 65, 1),
[I 2026-04-09 13:20:26,825] Trial 0 finished with value: 0.39640020901463235 and parameters: {'learning_rate': 0.07282414684217228, 'num_leaves': 227, 'max_depth': 40, 'feature_fraction': 0.8656358546687993, 'bagging_fraction': 0.9708953630128581, 'bagging_freq': 17, 'min_child_samples': 1019, 'min_data_in_leaf': 62, 'lambda_l1': 0.0012718019479997227, 'lambda_l2': 0.0064267811680307155, 'min_gain_to_split': 0.07299